In [31]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict, Annotated

In [32]:
def replace_value(current, new):
    return new if new is not None else current

In [35]:
class BatsmanState(TypedDict):
    runs: Annotated[int, replace_value]
    balls: Annotated[int, replace_value]
    fours: Annotated[int, replace_value]
    sixes: Annotated[int, replace_value]
    
    # Computed fields - also need reducers for parallel merge
    sr: Annotated[float, replace_value]
    bpb: Annotated[float, replace_value]
    boundry_precent: Annotated[float, replace_value]
    summary: Annotated[str, replace_value]

In [36]:
def calculate_sr(state: BatsmanState) -> BatsmanState:
    sr = (state['runs'] / state['balls']) * 100
    return {"sr":sr}

In [37]:
def calculate_bpb(state: BatsmanState) -> BatsmanState:
    bpb = state['balls'] / (state['fours']+state['sixes'])
    return {"bpb":bpb}

In [38]:
def calculate_boundry_precent(state: BatsmanState) -> BatsmanState:
    boundry_precent = (((state['fours']*4) + (state['sixes']*6)) / state['runs']) * 100
    return {"boundry_precent":boundry_precent}

In [39]:
def summary(state: BatsmanState) -> BatsmanState:
    summary=f"""
    Strikes Rate: {state['sr']} \n
    Balls per boundary: {state['bpb']} \n
    Boundry precent: {state['boundry_precent']} \n"""
    
    
    return {"summary":summary}

In [40]:
graph=StateGraph(BatsmanState)

graph.add_node('calculate_sr',calculate_sr)
graph.add_node('calculate_bpb',calculate_bpb)
graph.add_node('calculate_boundry_precent',calculate_boundry_precent)
graph.add_node('summary',summary)

In [41]:
graph.add_edge(START,'calculate_sr')
graph.add_edge(START,'calculate_bpb')
graph.add_edge(START,'calculate_boundry_precent')

graph.add_edge('calculate_sr','summary')
graph.add_edge('calculate_bpb','summary')
graph.add_edge('calculate_boundry_precent','summary')

graph.add_edge('summary',END)

workflow=graph.compile()

In [42]:
intial_state = {
    'runs': 100,
    'balls': 50,
    'fours': 6,
    'sixes': 4
}

workflow.invoke(intial_state)

{'runs': 100,
 'balls': 50,
 'fours': 6,
 'sixes': 4,
 'sr': 200.0,
 'bpb': 5.0,
 'boundry_precent': 48.0,
 'summary': '\n    Strikes Rate: 200.0 \n\n    Balls per boundary: 5.0 \n\n    Boundry precent: 48.0 \n'}